In [1]:
import ast
import numpy as np
import pandas as pd
import re

In [2]:
jobs_df = pd.read_excel("final_jobs_fixed.xlsx", engine="openpyxl")
jobs_df.head()

,id,title,company,location,country,salary_min,salary_max,contract_type,redirect_url,description,created
0,5753281217,Data Analyst (m/w/d),"{'display_name': 'STRABAG BRVZ GMBH', '__CLASS...","{'area': ['Österreich'], '__CLASS__': 'Adzuna:...",at,NaN,NaN,NaN,https://www.adzuna.at/land/ad/5753281217?se=_l...,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05T16:40:19Z
1,5753281216,Data Analyst (m/w/d),{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'display_name': 'Spittal an der Drau', '__CLA...",at,NaN,NaN,NaN,https://www.adzuna.at/land/ad/5753281216?se=_l...,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05T16:40:19Z
2,5762094132,Data Analyst - MS PowerBI (m/w/d),{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['Österreich', 'Wien'], '__CLASS__': ...",at,NaN,NaN,NaN,https://www.adzuna.at/land/ad/5762094132?se=_l...,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13T06:58:14Z
3,5727781690,Data Analyst (m/w/d),{'display_name': 'ÖSW Österreichisches Siedlun...,"{'display_name': 'Josefstadt, Wien', 'area': [...",at,NaN,NaN,permanent,https://www.adzuna.at/details/5727781690?utm_m...,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13T08:03:25Z
4,5758608143,Data Analyst,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'display_name': 'Österreich', 'area': ['Öster...",at,NaN,NaN,NaN,https://www.adzuna.at/details/5758608143?utm_m...,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10T17:18:19Z


In [3]:
jobs_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6718 entries, 0 to 6717
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             6718 non-null   int64  
 1   title          6718 non-null   object 
 2   company        6718 non-null   object 
 3   location       6718 non-null   object 
 4   country        6718 non-null   object 
 5   salary_min     2986 non-null   float64
 6   salary_max     2982 non-null   float64
 7   contract_type  1700 non-null   object 
 8   redirect_url   6718 non-null   object 
 9   description    6718 non-null   object 
 10  created        6718 non-null   object 
dtypes: float64(2), int64(1), object(8)
memory usage: 577.5+ KB


In [4]:
import re
import pandas as pd
from bs4 import BeautifulSoup


# ---------------------------------------------------------------------------
# KEYWORD DICTIONARIES
# Centralised here so adding a new language means editing one place only.
# Keys are the output label; values are the terms to match in the description.
# All terms must be lowercase — matching is done on a lowercased description.
# ---------------------------------------------------------------------------

# Work mode keywords by output label.
# Priority: Remote > Hybrid > Onsite (first match wins in the function below).
#
# Notes on specific terms:
#   "smart working"   (IT) — legally a flexible arrangement (2-3 days/home),
#                            NOT equivalent to full remote. Mapped to Hybrid.
#   "télétravail"     (FR) — typically means fully or near-fully remote in
#                            French job ads. Mapped to Remote.
#   "mobiles arbeiten"(DE) — "mobile working", usually partial office presence.
#                            Mapped to Hybrid.
#   "flexible"              — removed intentionally; too ambiguous (often refers
#                            to working hours, not location).
WORK_MODE_KEYWORDS = {
    "Remote": [
        "remote",           # EN
        "home-based",       # EN
        "home based",       # EN
        "fully remote",     # EN (explicit)
        "télétravail",      # FR
        "homeoffice",       # DE (one word, common spelling)
        "home office",      # DE (two words, also used in NL/BE)
        "lavoro da remoto", # IT
        "praca zdalna",     # PL
        "zdalnie",          # PL ("remotely")
        "thuiswerken",      # NL
        "thuis werken",     # NL (two words variant)
        "trabajo remoto",   # ES
    ],
    "Hybrid": [
        "hybrid",           # EN / DE / NL / PL (used across languages)
        "smart working",    # IT — flexible arrangement, NOT full remote
        "mobiles arbeiten", # DE — "mobile working", partial office
        "hybrydowo",        # PL — "hybridly"
        "hybride",          # FR
        "ibrido",           # IT (less common but present)
    ],
    "Onsite": [
        "onsite",           # EN
        "on-site",          # EN (hyphenated variant)
        "on site",          # EN (space variant)
        "office",           # EN — implies physical presence
        "in presenza",      # IT — "in person / on-site"
        "présentiel",       # FR — "in person"
        "vor ort",          # DE — "on location"
        "präsenz",          # DE — "presence"
        "op kantoor",       # NL — "at the office"
        "en oficina",       # ES — "in the office"
        "en presencial",    # ES — "in person"
    ],
}

# Seniority keywords by output label.
# Priority order: Intern > Junior > Senior > Middle > Unspecified.
# Intern is checked first to avoid e.g. "stage" or "praktikum" being
# misclassified as Junior due to other keywords in the same description.
SENIORITY_KEYWORDS = {
    "Intern": [
        "intern",           # EN
        "internship",       # EN
        "trainee",          # EN / DE
        "praktikum",        # DE — "internship"
        "praktikant",       # DE — "intern (person)"
        "werkstudent",      # DE — "working student"
        "absolvent",        # DE — "graduate / recent grad" (entry-level signal)
        "stagiaire",        # FR — "intern"
        "alternance",       # FR — apprenticeship contract (very common: ~20%)
        "apprentissage",    # FR — apprenticeship training
        "stage",            # FR / IT — "internship"
        "tirocinio",        # IT — "internship / traineeship"
        "apprendistato",    # IT — "apprenticeship"
        "neolaureato",      # IT — "recent graduate (m)"
        "neolaureata",      # IT — "recent graduate (f)"
        "stagiair",         # NL — "intern"
        "práctica",         # ES — "internship / practice"
        "becario",          # ES — "intern"
        "stażysta",         # PL — "intern"
    ],
    "Junior": [
        "junior",           # EN / DE / FR / NL / PL / ES
        "jr",               # EN (title abbreviation, checked in title field)
        "entry level",      # EN
        "entry-level",      # EN (hyphenated)
        "einsteiger",       # DE — "entry-level / beginner" (13.9% in DE data)
        "berufseinsteiger", # DE — "career starter"
        "débutant",         # FR — "beginner"
        "starter",          # NL — entry-level candidate (6.8%)
        "afgestudeerd",     # NL — "recently graduated"
    ],
    "Senior": [
        "senior",           # EN / DE / FR / NL / PL / ES
        "sr",               # EN (title abbreviation, checked in title field)
        "lead",             # EN — team/tech lead
        "principal",        # EN — principal-level role
        "expérimenté",      # FR — "experienced" (5.4%)
        "confirmé",         # FR — "confirmed / mid-senior level" (8.9%)
        "erfahren",         # DE — "experienced" (15%)
        "berufserfahrung",  # DE — "professional experience required" (19.3%)
        "führungskraft",    # DE — "executive / manager"
        "teamleiter",       # DE — "team leader"
        "ervaren",          # NL — "experienced" (10%)
        "experto",          # ES — "expert"
        "responsable",      # ES — "manager / lead" (9%)
        "anni di esperienza", # IT — "years of experience" (12.2%)
    ],
    "Middle": [
        "middle",           # EN
        "mid",              # EN / PL
        "mid-level",        # EN
        "confirmé",         # FR — sits between junior and senior in French usage
        "medior",           # NL — explicit mid-level label (2.5%)
        "livello intermedio", # IT — "intermediate level"
        "intermedio",       # IT — "intermediate"
    ],
}

# Contract type keywords by output label.
# This is a NEW field (extracted_contract) that complements the sparse
# Adzuna-provided contract_type column (which only contains "permanent"/"contract").
# Priority: Internship/Apprenticeship > Freelance > Fixed-term > Permanent.
CONTRACT_KEYWORDS = {
    "Internship": [
        "internship",       # EN
        "trainee",          # EN
        "stage",            # FR / IT
        "stagiaire",        # FR
        "alternance",       # FR — apprenticeship contract (~20% of FR listings)
        "apprentissage",    # FR
        "tirocinio",        # IT
        "apprendistato",    # IT
        "praktikum",        # DE
        "werkstudent",      # DE
        "stagiair",         # NL
        "práctica",         # ES
        "becario",          # ES
        "stażysta",         # PL
    ],
    "Freelance": [
        "freelance",        # EN / universal
        "b2b",              # PL — contractor billing model (27.6% in PL data)
        "kontrakt",         # PL — "contract" (B2B context)
        "partita iva",      # IT — VAT number = self-employed/freelance
        "zzp",              # NL — "self-employed" (zelfstandige zonder personeel)
        "selbstständig",    # DE — "self-employed"
        "freiberuflich",    # DE — "freelance"
        "autónomo",         # ES — "self-employed"
    ],
    "Fixed-term": [
        "fixed-term",       # EN
        "fixed term",       # EN
        "temporary",        # EN
        "cdd",              # FR — contrat à durée déterminée (3.2%)
        "intérim",          # FR — temporary agency work
        "befristet",        # DE — "limited / fixed-term" (9.5%)
        "tempo determinato",# IT — "fixed-term contract" (4.8%)
        "tijdelijk",        # NL — "temporary"
        "umowa zlecenie",   # PL — civil law contract (short-term)
        "temporal",         # ES — "temporary"
    ],
    "Permanent": [
        "permanent",        # EN
        "full-time",        # EN (used as proxy for permanent in EN ads)
        "cdi",              # FR — contrat à durée indéterminée (18.2%)
        "festanstellung",   # DE — "permanent / salaried position"
        "unbefristet",      # DE — "unlimited / permanent"
        "tempo indeterminato", # IT — "permanent contract" (18.8%)
        "vast",             # NL — "permanent" (29.4%)
        "umowa o pracę",    # PL — standard employment contract
        "indefinido",       # ES — "permanent contract"
    ],
}

# ---------------------------------------------------------------------------
# SKILL KEYWORDS — high-frequency skills (>=5% of jobs), named individually.
#
# WARNING Special case — "r":
#     Matching "r" as a plain substring hits 99.9% of descriptions (every word
#     containing the letter). It must be matched with a word boundary regex
#     (\br\b) instead — see the skills loop in the main function below.
#
# WARNING "scala" excluded intentionally:
#     Appears in 16.7% of descriptions but almost entirely as "scalable",
#     not the Scala programming language. Too noisy to include as-is.
#
# Frequency reference (% of all 6718 jobs in dataset):
#   sql 48% | excel 43% | git 37% | python 33% | power bi 32%
#   tableau 21% | r 15% | azure 13% | aws 11% | machine learning 9%
#   snowflake/databricks 7% | spark/dbt/gcp/looker 6-7% | bigquery 5%
# ---------------------------------------------------------------------------
SKILL_KEYWORDS = [
    # --- Core query & spreadsheet ---
    "python",           # 33.0%
    "sql",              # 48.1%
    "excel",            # 43.1%

    # --- BI & visualisation ---
    "tableau",          # 21.1%
    "power bi",         # 31.6%
    "looker",           # 6.1%

    # --- Cloud platforms ---
    "aws",              # 11.3%
    "azure",            # 13.2%
    "gcp",              # 6.2%

    # --- Modern data stack ---
    "snowflake",        # 7.4%
    "databricks",       # 7.4%
    "dbt",              # 6.5%
    "bigquery",         # 5.3%

    # --- Big data ---
    "spark",            # 6.8%

    # --- Stats & ML ---
    "r",                # ~14.7% with word boundary — see WARNING note above
    "machine learning", # 8.8%

    # --- Version control ---
    "git",              # 37.2%
]

# ---------------------------------------------------------------------------
# OTHER SKILL KEYWORDS — less common skills (<5% of jobs).
# If any of these are found in the description, a single "OTHER" token is
# appended to extracted_skills. No further detail is stored — the signal is
# simply that the job requires tools beyond the main stack.
#
# Includes: BI extras (DAX, Qlik, SAS), pipeline tools (Airflow, Kafka,
# PySpark), databases (NoSQL, Oracle, SQL Server, PostgreSQL, MySQL),
# Python libs (Pandas, Scikit, Jupyter), stats/ML niche (NLP, Stata),
# and other tools (VBA, Google Analytics).
# ---------------------------------------------------------------------------
OTHER_SKILL_KEYWORDS = [
    # BI extras — DAX is Power BI's formula language, rarely standalone
    "dax",              # 5.1%
    "qlik",             # 4.3% — catches both QlikView and QlikSense
    "sas",              # 4.4%

    # Pipeline & streaming
    "airflow",          # 3.6%
    "kafka",            # 2.4%
    "pyspark",          # 2.4% — Spark's Python API

    # Databases (beyond SQL as a language)
    "nosql",            # 3.5%
    "oracle",           # 3.4%
    "sql server",       # 2.7%
    "postgresql",       # 2.6%
    "mysql",            # 1.3%

    # Python data science libraries
    "pandas",           # 2.0%
    "scikit",           # 1.0% — catches scikit-learn
    "jupyter",          # 1.1%

    # Niche stats & ML
    "nlp",              # 1.0%
    "stata",            # 1.0%

    # Other
    "vba",              # 2.4% — Excel automation
    "google analytics", # 2.1%
]


# ---------------------------------------------------------------------------
# HELPER: HTML + whitespace cleaner
# Strips HTML tags (e.g. <li>, <b>, &amp;) and collapses all whitespace
# into single spaces. This prevents keywords like "power bi" from missing
# due to <b>power</b> bi or "power\nbi" in raw Adzuna descriptions.
# ---------------------------------------------------------------------------
def clean_text(raw):
    if not raw or not isinstance(raw, str):
        return ""
    text = BeautifulSoup(raw, "html.parser").get_text(separator=" ")
    text = re.sub(r'\s+', ' ', text)   # collapse tabs, newlines, double spaces
    return text.lower().strip()


# ---------------------------------------------------------------------------
# HELPER: keyword matcher
# Iterates a dict of {label: [keywords]} and returns the first matching label.
# Matching is done on the lowercased description (and optionally title).
# Returns `default` if nothing matches.
# ---------------------------------------------------------------------------
def match_keywords(text, keyword_dict, default="Unspecified"):
    for label, keywords in keyword_dict.items():
        for kw in keywords:
            if kw in text:
                return label
    return default


# ---------------------------------------------------------------------------
# MAIN EXTRACTION FUNCTION
# ---------------------------------------------------------------------------
def clean_and_extract_research_data(row):
    # Initialise all target fields
    country = "Unknown"
    salary = "Not Found"
    work_mode = "Unspecified"
    seniority = "Unspecified"
    extracted_contract = "Unspecified"
    skills = []

    # Strip HTML and normalise whitespace before lowercasing.
    # This prevents tags like <b>power</b> bi or newlines between words
    # from breaking multi-word keyword matches (e.g. "power bi", "machine learning").
    desc  = clean_text(row["description"]) if pd.notna(row["description"]) else ""
    title = clean_text(row["title"])       if pd.notna(row["title"])       else ""

    # Combine title + description into one string for keyword matching.
    # Title is prepended so that seniority signals in the job title (e.g.
    # "Senior Data Analyst") are captured even in short descriptions.
    full_text = title + " " + desc

    # --- Country ---
    if pd.notna(row["country"]):
        country = str(row["country"]).upper().strip()

    # --- Salary ---
    # Use structured min/max fields if both are present; otherwise fall back
    # to regex extraction from the description text.
    if pd.notna(row["salary_min"]) and pd.notna(row["salary_max"]):
        salary = f"{int(row['salary_min'])}-{int(row['salary_max'])}"
    else:
        # Regex matches currency symbols (£, $, €) or 'k' suffixes with
        # optional range indicators (-, ~, to).
        salary_pattern = r"([£$€]\s*\d+[\d,\s]*[-\s~toTO]?[\d,\s]*|\d+\s*[kK][-\s~toTO]?\d+\s*[kK])"
        salary_match = re.search(salary_pattern, desc)
        if salary_match:
            salary = salary_match.group(0).strip()

    # --- Skills ---
    # High-frequency skills are named individually in the output.
    # "r" is special-cased with a word boundary regex to avoid false positives:
    # plain substring matching hits 99.9% of descriptions (every word with "r").
    for skill in SKILL_KEYWORDS:
        if skill == "r":
            if re.search(r'\br\b', full_text):
                skills.append("R")
        else:
            if skill in full_text:
                skills.append(skill.upper())

    # Less common skills are collapsed into a single "OTHER" token.
    # This keeps the column clean while still flagging that additional
    # tooling is required beyond the main stack.
    if any(kw in full_text for kw in OTHER_SKILL_KEYWORDS):
        skills.append("OTHER")

    extracted_skills = ", ".join(skills) if skills else "None Mentioned"

    # --- Work Mode ---
    # Priority: Remote > Hybrid > Onsite.
    # "smart working" (IT) maps to Hybrid, not Remote — it is a legally
    # defined flexible arrangement, typically 2-3 days/week from home.
    # "télétravail" (FR) maps to Remote — implies full or near-full remote
    # in French job market conventions.
    work_mode = match_keywords(full_text, WORK_MODE_KEYWORDS, default="Unspecified")

    # --- Seniority ---
    # Priority: Intern > Junior > Senior > Middle.
    # Intern is checked first so internship-related terms (e.g. "stage",
    # "alternance") are not accidentally pulled into Junior/Senior buckets.
    seniority = match_keywords(full_text, SENIORITY_KEYWORDS, default="Unspecified")

    # --- Contract Type ---
    # Complements the Adzuna-provided contract_type column, which only
    # contains "permanent" / "contract" and is sparsely populated.
    # Priority: Internship > Freelance > Fixed-term > Permanent.
    extracted_contract = match_keywords(full_text, CONTRACT_KEYWORDS, default="Unspecified")

    # Return all fields as a Series to be assigned to new DataFrame columns.
    return pd.Series([
        country,
        salary,
        extracted_skills,
        work_mode,
        seniority,
        extracted_contract,
    ])


# ---------------------------------------------------------------------------
# APPLY TO DATAFRAME
# ---------------------------------------------------------------------------

# Normalise the created timestamp to timezone-naive UTC for consistent
# period arithmetic downstream.
jobs_df["created"] = (
    pd.to_datetime(jobs_df["created"], errors="coerce", utc=True)
    .dt.tz_localize(None)
)

# Apply extraction function row-by-row and assign results to new columns.
jobs_df[[
    "clean_country",
    "clean_salary",
    "extracted_skills",  # named high-frequency skills + "OTHER" if tail skills present
    "work_mode",
    "seniority_level",
    "extracted_contract",
]] = jobs_df.apply(clean_and_extract_research_data, axis=1)

# Convert creation timestamp to 'Year-Week' period string (e.g. '2026W24')
# for weekly trend analysis.
jobs_df["year_week"] = jobs_df["created"].dt.to_period("W").astype(str)


In [5]:
jobs_df['clean_country'].value_counts()

clean_country
FR    2095
GB    1968
DE     782
PL     591
NL     442
IT     271
ES     245
BE     191
CH      68
AT      65
Name: count, dtype: int64

In [6]:
# Create a pivot table to count the number of jobs ('id') by country and work mode
pivot_country = jobs_df.pivot_table(
    index="clean_country",
    columns="work_mode",
    values="id",
    aggfunc="count",
)

# Fill any missing/NaN values with 0 (indicating zero jobs found for that combination) 
pivot_country = pivot_country.fillna(0)


pivot_country

work_mode,Hybrid,Onsite,Remote,Unspecified
clean_country,,,,
AT,5,14,19,27
BE,30,25,32,104
CH,5,12,22,29
DE,174,127,314,167
ES,47,40,40,118
FR,96,195,688,1116
GB,476,320,303,869
IT,56,75,29,111
NL,77,51,72,242


In [7]:
# Create a pivot table to count the number of jobs ('id') by country and seniority level
pivot_seniority_leve = jobs_df.pivot_table(
    index="clean_country",
    columns="seniority_level",
    values="id",
    aggfunc="count",
)

# Fill any missing/NaN values with 0 to ensure clean numeric analysis
pivot_seniority_leve = pivot_seniority_leve.fillna(0)

pivot_seniority_leve

seniority_level,Intern,Junior,Middle,Senior,Unspecified
clean_country,,,,,
AT,36.0,3.0,1.0,16.0,9.0
BE,110.0,6.0,12.0,34.0,29.0
CH,45.0,2.0,0.0,12.0,9.0
DE,513.0,19.0,0.0,189.0,61.0
ES,147.0,8.0,1.0,59.0,30.0
FR,1503.0,24.0,0.0,393.0,175.0
GB,826.0,109.0,4.0,609.0,420.0
IT,200.0,14.0,0.0,40.0,17.0
NL,263.0,18.0,37.0,55.0,69.0


In [8]:
# Group the dataset by the 'year_week' column and calculate the total number of records (rows) per week
jobs_df.groupby('year_week').size()

year_week
2022-06-27/2022-07-03       4
2022-08-29/2022-09-04       1
2022-10-24/2022-10-30       1
2022-11-28/2022-12-04       2
2023-02-13/2023-02-19       3
                         ... 
2026-05-18/2026-05-24     633
2026-05-25/2026-05-31     623
2026-06-01/2026-06-07    1029
2026-06-08/2026-06-14    1279
2026-06-15/2026-06-21     337
Length: 113, dtype: int64

In [9]:
# Expand the comma-separated skills into individual rows
# .assign(): Split the comma-separated string 'extracted_skills' into Python lists (e.g., "PYTHON, SQL" -> ["PYTHON", "SQL"])
# .explode(): Flatten the lists so each skill gets its own dedicated row, duplicating other row values accordingly
exploded_df = jobs_df.assign(extracted_skills=jobs_df["extracted_skills"].str.split(", ")).explode("extracted_skills")
# Use pivot_table to achieve the same cross-tabulation frequency count
pivot_skills = exploded_df.pivot_table(
    index="country", 
    columns="extracted_skills", 
    values="id", 
    aggfunc="count"
)

# Fill NaN with 0 since missing combination means 0 occurrences
pivot_skills = pivot_skills.fillna(0)
# Convert all counts from float to integer for a cleaner, non-decimal display
pivot_skills = pivot_skills.astype(int)

pivot_skills

extracted_skills,AWS,AZURE,BIGQUERY,DATABRICKS,DBT,EXCEL,GCP,GIT,LOOKER,MACHINE LEARNING,None Mentioned,OTHER,POWER BI,PYTHON,R,SNOWFLAKE,SPARK,SQL,TABLEAU
country,,,,,,,,,,,,,,,,,,,
at,9,15,3,6,6,24,3,28,2,8,5,22,22,23,9,3,7,37,11
be,23,31,4,29,12,81,9,86,6,20,16,54,68,71,12,11,13,98,39
ch,12,13,7,10,7,29,5,28,8,6,1,21,22,34,13,12,9,46,15
de,96,208,46,80,88,208,33,412,49,68,53,328,319,286,128,56,55,482,121
es,29,23,25,15,19,156,9,90,43,23,16,103,78,100,41,16,12,155,63
fr,216,198,115,139,145,907,207,928,116,178,147,733,670,689,475,154,135,1006,743
gb,207,202,71,112,104,889,81,493,93,170,562,395,472,497,124,116,110,721,224
it,24,30,11,17,9,159,15,116,22,29,11,87,106,90,41,16,18,132,50
nl,32,61,19,45,28,126,12,121,18,35,116,99,131,152,72,26,36,185,41


In [10]:
skills_count = exploded_df["extracted_skills"].value_counts().reset_index()
skills_count.columns = ["skill", "count"]
skills_count


,skill,count
0,SQL,3235
1,EXCEL,2896
2,GIT,2505
3,PYTHON,2217
4,POWER BI,2123
5,OTHER,2082
6,TABLEAU,1421
7,R,986
8,None Mentioned,959
9,AZURE,888


In [11]:
# Define a function to parse city names from a location/display name string
def extract_city_simple(location_str):
    # Check for invalid types (non-strings) or empty/whitespace-only entries
    if not isinstance(location_str, str) or not location_str.strip():
        return "Unknown"

    # Use regular expression to extract the value associated with the 'display_name' key    
    match = re.search(r"'display_name':\s*'([^']+)'", location_str)
    if match:

        # Return the captured group (the actual city/location name)
        return match.group(1)
    return "Unknown"


# Standardize country codes by converting to uppercase, stripping whitespace, and filling NaNs
jobs_df["clean_country"] = (
    jobs_df["country"].str.upper().str.strip().fillna("UNKNOWN")
)
# Apply the custom extraction function row-by-row to extract clean city names
jobs_df["clean_city"] = jobs_df["location"].apply(extract_city_simple)



# Create a baseline aggregation table counting job volume by country and city 
city_table = (
    jobs_df.groupby(["clean_country", "clean_city"])
    .size()
    .reset_index(name="job_count")
)

# Calculate the percentage market share of each city within its respective country
# .transform() scales the group total back to the original dataframe size to allow direct division
city_table["percentage_in_country"] = city_table.groupby("clean_country")[
    "job_count"
].transform(lambda x: (x / x.sum()) * 100)

# Extract the top 3 cities with the highest job counts for each country 
final_table = (
    # Sort primarily by country (A-Z) and secondarily by job count descending (highest first)
    city_table.sort_values(
        ["clean_country", "job_count"], ascending=[True, False]
    )
    # Group by country again to isolate each regional market
    .groupby("clean_country")
    # Slice the top 3 rows for each country group
    .head(3)
)

# Round the calculated percentage values to 2 decimal places for cleaner reporting 
final_table["percentage_in_country"] = final_table[
    "percentage_in_country"
].round(2)


# Extract company names from the nested company/display_name field
def extract_company_display_name(company_value):
    if isinstance(company_value, dict):
        return company_value.get("display_name", "Unknown")

    if pd.isna(company_value):
        return "Unknown"

    company_text = str(company_value)
    try:
        company_dict = ast.literal_eval(company_text)
        if isinstance(company_dict, dict):
            return company_dict.get("display_name", "Unknown")
    except (ValueError, SyntaxError):
        pass

    match = re.search(r"'display_name':\s*'([^']+)'", company_text)
    return match.group(1) if match else "Unknown"


jobs_df["clean_company"] = jobs_df["company"].apply(extract_company_display_name)

 
print(final_table)


     clean_country                          clean_city  job_count  \
15              AT                          Österreich         20   
13              AT                    Wien, Österreich         16   
4               AT                  Innere Stadt, Wien          9   
21              BE                              België        127   
23              BE          Brussel, Brussel Hoofdstad         13   
36              BE                            Mechelen          6   
61              CH                             Schweiz         22   
66              CH                              Zürich         15   
46              CH               Bern, Bern-Mittelland          5   
142             DE                         Deutschland        122   
113             DE                 Berlin, Deutschland         82   
202             DE                Hamburg, Deutschland         22   
389             ES                              España         89   
394             ES                

In [12]:
def extract_company_display_name(company_value):
    if pd.isna(company_value):
        return "Unknown"

    company_text = str(company_value)

    match = re.search(r"'display_name':\s*'([^']+)'", company_text)

    if match:
        return match.group(1)

    return "Unknown"


jobs_df["clean_company"] = jobs_df["company"].apply(extract_company_display_name)

jobs_df[["company", "clean_company"]].head()

,company,clean_company
0,"{'display_name': 'STRABAG BRVZ GMBH', '__CLASS...",STRABAG BRVZ GMBH
1,{'__CLASS__': 'Adzuna::API::Response::Company'...,STRABAG BRVZ GMBH
2,{'__CLASS__': 'Adzuna::API::Response::Company'...,Baustoff + Metall Gesellschaft m.b.H. Österrei...
3,{'display_name': 'ÖSW Österreichisches Siedlun...,ÖSW Österreichisches Siedlungswerk Gemeinnützi...
4,{'__CLASS__': 'Adzuna::API::Response::Company'...,Hitapps


In [13]:
jobs_df.sample(20)

,id,title,company,location,country,salary_min,salary_max,contract_type,redirect_url,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,extracted_contract,year_week,clean_city,clean_company
1691,5765386397,Data Engineer Databricks,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['France', 'Ile-de-France', 'Paris'],...",fr,NaN,NaN,NaN,https://www.adzuna.fr/land/ad/5765386397?se=9C...,Êtes-vous prêt à devenir un expert de la Data ...,2026-06-16 11:48:06,FR,Not Found,"PYTHON, SQL, EXCEL, AZURE, DATABRICKS, DBT, SP...",Unspecified,Intern,Unspecified,2026-06-15/2026-06-21,"Paris, Ile-de-France",Datatorii
6692,5661027115,Solutions Analyst,"{'display_name': 'RELX INC', '__CLASS__': 'Adz...","{'display_name': 'Farringdon, Exeter', '__CLAS...",gb,38230.53,38230.53,NaN,https://www.adzuna.co.uk/jobs/details/56610271...,Solutions Analyst\nAbout our team:\nLexisNexis...,2026-03-10 21:59:12,GB,38230-38230,"EXCEL, MACHINE LEARNING",Hybrid,Intern,Unspecified,2026-03-09/2026-03-15,"Farringdon, Exeter",RELX INC
3964,5719626229,Data Analyst,"{'display_name': 'Verita HR', '__CLASS__': 'Ad...",{'__CLASS__': 'Adzuna::API::Response::Location...,pl,300000.00,348000.00,NaN,https://www.adzuna.pl/details/5719626229?utm_m...,Czym będziesz się zajmować?\nClient:\ninternat...,2026-05-05 21:21:28,PL,300000-348000,"PYTHON, SQL, LOOKER, GCP, GIT",Remote,Intern,Freelance,2026-05-04/2026-05-10,Polska,Verita HR
479,5757774185,Consumer Data Analyst F/H,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['France', 'Auvergne-Rhône-Alpes', 'H...",fr,NaN,NaN,NaN,https://www.adzuna.fr/details/5757774185?utm_m...,Description du poste\nEn tant que Consumer Dat...,2026-06-10 00:38:25,FR,Not Found,"POWER BI, MACHINE LEARNING",Unspecified,Unspecified,Unspecified,2026-06-08/2026-06-14,"Annecy, Haute-Savoie",Salomon
3737,5761960841,Applied LLM Data Scientist,"{'display_name': 'ING', '__CLASS__': 'Adzuna::...","{'display_name': 'Nederland', 'area': ['Nederl...",nl,NaN,NaN,NaN,https://www.adzuna.nl/details/5761960841?utm_m...,Applied LLM Data Scientist\nAs an Applied GenA...,2026-06-13 05:33:44,NL,Not Found,"PYTHON, SQL, BIGQUERY, SPARK, MACHINE LEARNING...",Hybrid,Intern,Unspecified,2026-06-08/2026-06-14,Nederland,ING
5988,5735549168,Project Director - Controls Analytics,{'__CLASS__': 'Adzuna::API::Response::Company'...,{'__CLASS__': 'Adzuna::API::Response::Location...,gb,74335.50,74335.50,NaN,https://www.adzuna.co.uk/jobs/details/57355491...,Job Description\nStart Here Grow Here\nOur awa...,2026-05-20 21:53:28,GB,74335-74335,None Mentioned,Unspecified,Senior,Unspecified,2026-05-18/2026-05-24,UK,AECOM
3188,5743147170,Data Analyst Junior,"{'display_name': 'Synergie Italia S.p.a.', '__...","{'display_name': 'Cuneo, Provincia di Cuneo', ...",it,NaN,NaN,permanent,https://www.adzuna.it/details/5743147170?utm_m...,"La Divisione ICT Corporate di Synergie Italia,...",2026-05-28 07:41:34,IT,Not Found,"PYTHON, SQL, EXCEL, POWER BI, R, MACHINE LEARNING",Unspecified,Junior,Permanent,2026-05-25/2026-05-31,"Cuneo, Provincia di Cuneo",Synergie Italia S.p.a.
6408,5742727682,Senior Performance Analyst,"{'display_name': '. Netcompany UK Limited', '_...",{'__CLASS__': 'Adzuna::API::Response::Location...,gb,43453.02,43453.02,NaN,https://www.adzuna.co.uk/jobs/details/57427276...,Company Description\nAre you ready to join the...,2026-05-27 22:42:13,GB,43453-43453,"SQL, TABLEAU, POWER BI, GIT, OTHER",Hybrid,Senior,Unspecified,2026-05-25/2026-05-31,"Birmingham, West Midlands",. Netcompany UK Limited
4297,5371654344,Data and AI Solutions Analyst,{'__CLASS__': 'Adzuna::API::Response::Company'...,"{'area': ['Polska', 'mazowieckie', 'Warszawa']...",pl,NaN,NaN,NaN,https://www.adzuna.pl/details/5371654344?utm_m...,Data and AI Solutions Analyst\nData and AI Sol...,2025-08-28 12:53:04,PL,Not Found,"PYTHON, SQL, AZURE, DATABRICKS, SPARK, GIT, OTHER",Remote,Senior,Unspecified,2025-08-25/2025-08-31,"Warszawa, mazowieckie",EY
2832,5757056768,Upskilling 

In [14]:
jobs_df.columns

Index(['id', 'title', 'company', 'location', 'country', 'salary_min',
       'salary_max', 'contract_type', 'redirect_url', 'description', 'created',
       'clean_country', 'clean_salary', 'extracted_skills', 'work_mode',
       'seniority_level', 'extracted_contract', 'year_week', 'clean_city',
       'clean_company'],
      dtype='object')

In [15]:
jobs_df_clean = jobs_df.copy()

In [16]:
jobs_df_clean = jobs_df_clean.drop(columns=
                       ["company", 
                        "location", 
                        "salary_min",
                        "salary_max",
                        "country",
                        "redirect_url"]
                       )
jobs_df_clean.head(5)

,id,title,contract_type,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,extracted_contract,year_week,clean_city,clean_company
0,5753281217,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,"GIT, OTHER",Remote,Intern,Unspecified,2026-06-01/2026-06-07,Österreich,STRABAG BRVZ GMBH
1,5753281216,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,"GIT, OTHER",Remote,Intern,Unspecified,2026-06-01/2026-06-07,Spittal an der Drau,STRABAG BRVZ GMBH
2,5762094132,Data Analyst - MS PowerBI (m/w/d),NaN,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13 06:58:14,AT,Not Found,"SQL, OTHER",Unspecified,Intern,Freelance,2026-06-08/2026-06-14,"Wien, Österreich",Baustoff + Metall Gesellschaft m.b.H. Österrei...
3,5727781690,Data Analyst (m/w/d),permanent,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13 08:03:25,AT,Not Found,SQL,Onsite,Intern,Internship,2026-05-11/2026-05-17,"Josefstadt, Wien",ÖSW Österreichisches Siedlungswerk Gemeinnützi...
4,5758608143,Data Analyst,NaN,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10 17:18:19,AT,Not Found,"PYTHON, SQL, TABLEAU, LOOKER",Unspecified,Unspecified,Unspecified,2026-06-08/2026-06-14,Österreich,Hitapps


In [17]:
jobs_df_clean.nunique().sort_values(ascending=False)


id                    6718
created               5863
description           5592
title                 4403
clean_company         3018
clean_salary          2027
extracted_skills      1586
clean_city            1378
year_week              113
clean_country           10
seniority_level          5
extracted_contract       5
work_mode                4
contract_type            2
dtype: int64

In [18]:
jobs_df_clean.describe(include="object")

,title,contract_type,description,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,extracted_contract,year_week,clean_city,clean_company
count,6718,1700,6718,6718,6718,6718,6718,6718,6718,6718,6718,6718
unique,4403,2,5592,10,2027,1586,4,5,5,113,1378,3018
top,Data Analyst,permanent,Too Many Requests,FR,Not Found,None Mentioned,Unspecified,Intern,Unspecified,2026-06-08/2026-06-14,"London, UK",Externatic
freq,325,998,465,2095,3420,959,2993,3960,3536,1279,550,235


In [19]:
jobs_df_clean.columns

Index(['id', 'title', 'contract_type', 'description', 'created',
       'clean_country', 'clean_salary', 'extracted_skills', 'work_mode',
       'seniority_level', 'extracted_contract', 'year_week', 'clean_city',
       'clean_company'],
      dtype='object')

In [20]:
jobs_df_clean["title"].unique()

array(['Data Analyst (m/w/d)', 'Data Analyst - MS PowerBI (m/w/d)',
       'Data Analyst', ..., 'Executive, Data Excellence',
       'Founding Product Designer', 'Business Analyst (Ref: 18005)'],
      shape=(4403,), dtype=object)

In [21]:
jobs_df_clean["extracted_contract"].unique()

array(['Unspecified', 'Freelance', 'Internship', 'Permanent',
       'Fixed-term'], dtype=object)

In [22]:
jobs_df_clean["clean_country"].unique()

array(['AT', 'BE', 'FR', 'DE', 'IT', 'NL', 'PL', 'ES', 'CH', 'GB'],
      dtype=object)

In [23]:
jobs_df_clean["work_mode"].unique()

array(['Remote', 'Unspecified', 'Onsite', 'Hybrid'], dtype=object)

In [24]:
jobs_df_clean["seniority_level"].unique()

array(['Intern', 'Unspecified', 'Junior', 'Senior', 'Middle'],
      dtype=object)

In [25]:
jobs_df_clean["clean_company"].unique()

array(['STRABAG BRVZ GMBH',
       'Baustoff + Metall Gesellschaft m.b.H. Österreich Hauptsitz',
       'ÖSW Österreichisches Siedlungswerk Gemeinnützige Wohnungs AG',
       ..., 'ServiceTrade', 'WPP Media WPP', 'Fifth Dimension'],
      shape=(3018,), dtype=object)

In [26]:
jobs_df_clean["extracted_contract"].value_counts(dropna=False)

extracted_contract
Unspecified    3536
Internship     1467
Permanent       922
Freelance       511
Fixed-term      282
Name: count, dtype: int64

In [27]:
jobs_df_clean["clean_country"].value_counts(dropna=False)

clean_country
FR    2095
GB    1968
DE     782
PL     591
NL     442
IT     271
ES     245
BE     191
CH      68
AT      65
Name: count, dtype: int64

In [28]:
jobs_df_clean["work_mode"].value_counts(dropna=False)

work_mode
Unspecified    2993
Remote         1746
Hybrid         1087
Onsite          892
Name: count, dtype: int64

In [29]:
jobs_df_clean["seniority_level"].value_counts(dropna=False)

seniority_level
Intern         3960
Senior         1538
Unspecified     928
Junior          235
Middle           57
Name: count, dtype: int64

In [30]:
jobs_df_clean["clean_company"].value_counts(dropna=False)

clean_company
Externatic                       235
Unknown                          154
Smart Future Campus GmbH         105
Collective.work                  105
ITOL Recruit                      88
                                ... 
Usercentrics                       1
FullStack Talents                  1
MCG Memmingen GmbH                 1
EFW – Elbe Flugzeugwerke GmbH      1
Fifth Dimension                    1
Name: count, Length: 3018, dtype: int64

In [31]:
jobs_df_clean.duplicated().sum()

np.int64(0)

In [32]:
jobs_df_clean["title_clean"] = (
    jobs_df_clean["title"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace("-", " ", regex=False)
    .str.replace(r"\(.*?\)", "", regex=True)
    .str.strip()
)

In [33]:
jobs_df_clean["title_clean"].value_counts().head(20)

title_clean
data analyst                                                                                                    479
senior data analyst                                                                                             137
data engineer                                                                                                   127
data analyst trainee                                                                                             61
upskilling als controller : daten management mit ki, azure & power bi – für den schnellen wiedereinstieg         39
analytics engineer                                                                                               37
referent der geschäftsführung : datenanalyse für unternehmenssteuerung & reporting mit sql, azure & power bi     33
deine chance: datenmanagement upskilling   sql, microsoft azure & power bi für den schnellen wiedereinstieg      33
data analyst f/h                                            

In [34]:
jobs_df_clean[["title", "title_clean"]].head(20)

,title,title_clean
0,Data Analyst (m/w/d),data analyst
1,Data Analyst (m/w/d),data analyst
2,Data Analyst - MS PowerBI (m/w/d),data analyst ms powerbi
3,Data Analyst (m/w/d),data analyst
4,Data Analyst,data analyst
5,(Junior) Data Analyst,data analyst
6,Senior Data Analyst:in,senior data analyst:in
7,Supply Chain Data Analyst*in,supply chain data analyst*in
8,Data Analyst (Gaming Industry),data analyst
9,Data Analyst / Data Manager (m/w/d),data analyst / data manager


In [35]:
jobs_df_clean["job_category"] = "other"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("data analyst", na=False),
    "job_category"
] = "data analyst"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("engineer", na=False),
    "job_category"
] = "data engineer"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("scientist", na=False),
    "job_category"
] = "data scientist"

jobs_df_clean.loc[
    jobs_df_clean["title_clean"].str.contains("business", na=False),
    "job_category"
] = "business analyst"

In [36]:
jobs_df_clean["job_category"].value_counts().head(20)

job_category
other               2819
data analyst        2496
data engineer        754
business analyst     499
data scientist       150
Name: count, dtype: int64

In [37]:
jobs_df_clean["clean_salary"] = (
    jobs_df_clean["clean_salary"]
    .replace("not found", pd.NA)
    .astype("string")
    .str.replace(r"[€$£¥₹]", "", regex=True)
    .str.replace(r"\b(eur|usd|gbp)\b", "", regex=True, case=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [38]:
jobs_df_clean[["clean_salary"]].head(20)

,clean_salary
0,Not Found
1,Not Found
2,Not Found
3,Not Found
4,Not Found
5,Not Found
6,Not Found
7,Not Found
8,Not Found
9,Not Found


In [39]:
jobs_df_clean["clean_salary"] = (
    jobs_df_clean["clean_salary"]
    .astype("string")
    .str.replace(r"[€$£¥₹]", "", regex=True)
    .str.strip()
)

salary_split = jobs_df_clean["clean_salary"].str.split("-", n=1, expand=True)

jobs_df_clean["min_salary"] = salary_split[0].str.strip()
jobs_df_clean["max_salary"] = salary_split[1].str.strip()

jobs_df_clean.loc[
    jobs_df_clean["clean_salary"] == "not found",
    ["min_salary", "max_salary"]
] = "not found"

In [40]:
jobs_df_clean[["clean_salary", "min_salary", "max_salary"]].head(20)

,clean_salary,min_salary,max_salary
0,Not Found,Not Found,<NA>
1,Not Found,Not Found,<NA>
2,Not Found,Not Found,<NA>
3,Not Found,Not Found,<NA>
4,Not Found,Not Found,<NA>
5,Not Found,Not Found,<NA>
6,Not Found,Not Found,<NA>
7,Not Found,Not Found,<NA>
8,Not Found,Not Found,<NA>
9,Not Found,Not Found,<NA>


In [41]:

import numpy as np

jobs_df_clean["currency"] = np.where(
    jobs_df_clean["clean_country"].str.lower() == "pl",
    "PLN",
    np.where(
        jobs_df_clean["clean_country"].str.lower() == "gb",
        "GBP",
        "EUR"
    )
)

In [42]:
jobs_df_clean[["clean_country", "currency"]].sample(20)

,clean_country,currency
2659,DE,EUR
4371,PL,PLN
4041,PL,PLN
2194,FR,EUR
2320,FR,EUR
5635,GB,GBP
1380,FR,EUR
4516,ES,EUR
4280,PL,PLN
1873,FR,EUR


In [43]:
exchange_rate_map = {
    "EUR": 1.0,
    "PLN": 0.23,
    "GBP": 1.16
}

jobs_df_clean["exchange_rate"] = jobs_df_clean["currency"].map(exchange_rate_map)

In [44]:
min_num = pd.to_numeric(jobs_df_clean["min_salary"], errors="coerce")
max_num = pd.to_numeric(jobs_df_clean["max_salary"], errors="coerce")

jobs_df_clean["MIN_EUR"] = min_num * jobs_df_clean["exchange_rate"]
jobs_df_clean["MAX_EUR"] = max_num * jobs_df_clean["exchange_rate"]

In [45]:
jobs_df_clean

,id,title,contract_type,description,created,clean_country,clean_salary,extracted_skills,work_mode,seniority_level,...,clean_city,clean_company,title_clean,job_category,min_salary,max_salary,currency,exchange_rate,MIN_EUR,MAX_EUR
0,5753281217,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,"GIT, OTHER",Remote,Intern,...,Österreich,STRABAG BRVZ GMBH,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
1,5753281216,Data Analyst (m/w/d),NaN,Bei STRABAG bauen rund 89.000 Menschen an mehr...,2026-06-05 16:40:19,AT,Not Found,"GIT, OTHER",Remote,Intern,...,Spittal an der Drau,STRABAG BRVZ GMBH,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
2,5762094132,Data Analyst - MS PowerBI (m/w/d),NaN,Baustoff + Metall ist ein hoch spezialisierter...,2026-06-13 06:58:14,AT,Not Found,"SQL, OTHER",Unspecified,Intern,...,"Wien, Österreich",Baustoff + Metall Gesellschaft m.b.H. Österrei...,data analyst ms powerbi,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
3,5727781690,Data Analyst (m/w/d),permanent,Data Analyst (m/w/d)\nJobs fürs Leben\nAuf der...,2026-05-13 08:03:25,AT,Not Found,SQL,Onsite,Intern,...,"Josefstadt, Wien",ÖSW Österreichisches Siedlungswerk Gemeinnützi...,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
4,5758608143,Data Analyst,NaN,"What are you working on?\nGenres: Casual, Puzz...",2026-06-10 17:18:19,AT,Not Found,"PYTHON, SQL, TABLEAU, LOOKER",Unspecified,Unspecified,...,Österreich,Hitapps,data analyst,data analyst,Not Found,<NA>,EUR,1.00,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6713,5734024897,"Executive, Data Excellence",NaN,About WPP Media\nWPP is the trusted growth par...,2026-05-19 13:14:36,GB,29787-29787,"SQL, EXCEL, POWER BI",Hybrid,Intern,...,"London, UK",WPP Media WPP,"executive, data excellence",other,29787,29787,GBP,1.16,34552.92,34552.92
6714,5731913923,Principal Product Marketing Manager,contract,"Sprinklr is the definitive, AI-native platform...",2026-05-17 01:20:20,GB,76986-76986,"EXCEL, AWS, GIT",Unspecified,Intern,...,"London, UK",Sprinklr,principal product marketing manager,other,76986,76986,GBP,1.16,89303.76,89303.76
6715,5672309969,Founding Product Designer,NaN,The Opportunity\nFifth Dimension is the most p...,2026-03-19 20:25:48,GB,70612-70612,None Mentioned,Hybrid,Intern,...,"Oxford Circus, Central London",Fifth Dimension,founding product designer,other,70612,70612,GBP,1.16,81909.92,81909.92
6716,5749733223,Business Analyst (Ref: 18005),permanent,About the job\nJob summary\nThis position is b...,2026-06-02 22:59:45,GB,51993-51993,"EXCEL, GIT",Remote,Intern,...,"London, UK",. Ministry of Justice,business analyst,business analyst,51993,51993,GBP,1.16,60311.88,60311.88


In [46]:
jobs_df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6718 entries, 0 to 6717
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id                  6718 non-null   int64         
 1   title               6718 non-null   object        
 2   contract_type       1700 non-null   object        
 3   description         6718 non-null   object        
 4   created             6718 non-null   datetime64[ns]
 5   clean_country       6718 non-null   object        
 6   clean_salary        6718 non-null   string        
 7   extracted_skills    6718 non-null   object        
 8   work_mode           6718 non-null   object        
 9   seniority_level     6718 non-null   object        
 10  extracted_contract  6718 non-null   object        
 11  year_week           6718 non-null   object        
 12  clean_city          6718 non-null   object        
 13  clean_company       6718 non-null   object      

In [47]:
jobs_df_clean['MIN_EUR'].min()

np.float64(0.0)

In [48]:
jobs_df_clean['MIN_EUR'].max()

np.float64(600000.0)

In [49]:
jobs_df_clean['MAX_EUR'].min()

np.float64(1.16)

In [50]:
jobs_df_clean['MAX_EUR'].max()

np.float64(600000.0)

In [51]:
jobs_df_clean.to_csv("jobs_clean.csv")